# VOC output plots

Line plots over time of all VOC-related DEMENTpy output files, with the runs listed in `RUNS` shown
**side by side** (one column per run, shared y-axis), so model versions run with the same seed can be compared.

Every execution saves its figures into a new, timestamped folder under `plots/`
(e.g. `plots/20261013-142501__2026101201_vs_2026101301/`), so earlier plots are never overwritten.

In [ ]:
from pathlib import Path
from datetime import datetime
import re

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd

%matplotlib inline

## Settings
`RUNS`: output folders to compare, left to right. Leave empty (`[]`) to use every run that has VOC output.
`LABELS`: optional panel titles per run, e.g. to name the model version.

In [ ]:
OUTPUT_DIR = Path("output")
RUNS = ["2026101201", "2026101301"]
LABELS = {
    # "2026101201": "v1.0.0",
    # "2026101301": "v1.0.1 - separate VOC transporters",
}

SAVE_PLOTS = True
PLOT_ROOT = Path("plots")
PANEL_W, PANEL_H = 5.5, 3.8  # size of each panel (inches)

available = sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_dir() and any(p.glob("*VOC*.csv")))
print("Runs with VOC output:", available)
if not RUNS:
    RUNS = available
missing = [r for r in RUNS if not (OUTPUT_DIR / r).is_dir()]
assert not missing, f"Run folders not found: {missing}"

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
PLOT_DIR = PLOT_ROOT / f"{stamp}__{'_vs_'.join(RUNS)}"
print("Comparing:", RUNS)

## Helpers

In [ ]:
def natural_key(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def label(run):
    return LABELS.get(run, run)


def read_runtime(run):
    return pd.read_csv(OUTPUT_DIR / run / "Runtime.csv", index_col=0).iloc[:, 0]


def n_days(run):
    rt = read_runtime(run)
    return int(rt["end_time"] * rt["pulse"])


def load_timeseries(run, fname, wide):
    """Load an output file as a dataframe with one row per day (0 = initial state).

    wide=True: rows are pools/taxa and columns are days (e.g. SubstratesSeries_VOC.csv).
    wide=False: rows are days (e.g. VOC_uptake.csv).
    Column headers in the wide files do not carry the day, so days are taken from the column position.
    """
    path = OUTPUT_DIR / run / fname
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0)
    if wide:
        df = df.T
    df = df.reset_index(drop=True).apply(pd.to_numeric, errors="coerce")
    # MonomersSeries.csv is appended twice per day by output.py -> keep one value per day
    if len(df) == 2 * n_days(run) + 1:
        df = df.iloc[::2].reset_index(drop=True)
    df.index.name = "day"
    return df


def save(fig, name):
    if SAVE_PLOTS:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(PLOT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


def side_by_side_axes(n_rows=1, sharey="row"):
    fig, axes = plt.subplots(n_rows, len(RUNS), figsize=(PANEL_W * len(RUNS), PANEL_H * n_rows),
                             sharex=True, sharey=sharey, squeeze=False)
    for ax, run in zip(axes[0], RUNS):
        ax.set_title(label(run))
    for ax in axes[-1]:
        ax.set_xlabel("day")
    return fig, axes


def mark_missing(ax, fname):
    ax.text(0.5, 0.5, f"{fname}\nnot found", ha="center", va="center", transform=ax.transAxes, color="grey")

## Parameter differences between the runs
Runtime parameters that differ between the selected runs (empty if all are identical).

In [ ]:
runtimes = pd.DataFrame({label(r): read_runtime(r) for r in RUNS})
differs = runtimes.nunique(axis=1, dropna=False) > 1
runtimes[differs] if differs.any() else print("All runtime parameters are identical.")

## VOC pools and fluxes
One line per VOC type; a VOC has the same colour in every panel.

In [ ]:
VOC_FILES = [
    # file,                      wide,  title,                          y label,      VOC rows only
    ("SubstratesSeries_VOC.csv", True,  "VOC substrate pool in litter", "C",          False),
    ("VOC_litter_input.csv",     False, "VOC release from litter",      "C per day",  False),
    ("MonomersSeries.csv",       True,  "VOC monomer pool",             "C",          True),
    ("VOC_uptake.csv",           False, "Total VOC uptake",             "C per day",  False),
]


def plot_voc_file(fname, wide, title, ylabel, voc_only):
    data = {run: load_timeseries(run, fname, wide) for run in RUNS}
    if voc_only:
        data = {r: (d.filter(regex=r"^VOC\d+$") if d is not None else None) for r, d in data.items()}
    if all(d is None or d.empty for d in data.values()):
        print(f"{fname}: no VOC data in any run, skipped")
        return
    names = sorted({c for d in data.values() if d is not None for c in d.columns}, key=natural_key)
    colors = {n: f"C{i % 10}" for i, n in enumerate(names)}

    fig, axes = side_by_side_axes()
    for ax, run in zip(axes[0], RUNS):
        d = data[run]
        if d is None or d.empty:
            mark_missing(ax, fname if d is None else "VOC rows")
            continue
        for c in d.columns:
            ax.plot(d.index, d[c], color=colors[c], label=c)
    axes[0, 0].set_ylabel(ylabel)
    handles = [Line2D([], [], color=colors[n], label=n) for n in names]
    fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    fig.suptitle(f"{title}  ({fname})")
    fig.tight_layout()
    save(fig, Path(fname).stem + ("_VOC" if voc_only else ""))
    plt.show()


for spec in VOC_FILES:
    plot_voc_file(*spec)

## VOC uptake per taxon
One row per VOC type, one line per taxon that takes up that VOC at some point in the run.
A taxon has the same colour in every panel.

In [ ]:
def plot_taxon_uptake():
    files = sorted({p.name for r in RUNS for p in (OUTPUT_DIR / r).glob("VOC*_uptake_TaxonSeries.csv")},
                   key=natural_key)
    if not files:
        print("No VOC*_uptake_TaxonSeries.csv files found, skipped")
        return
    n_taxa = max(int(read_runtime(r)["n_taxa"]) for r in RUNS)
    cmap = plt.get_cmap("turbo", n_taxa)

    def color(taxon):
        return cmap((int(re.sub(r"\D", "", taxon)) - 1) % n_taxa)

    fig, axes = side_by_side_axes(n_rows=len(files))
    for row, fname in zip(axes, files):
        active = set()
        for ax, run in zip(row, RUNS):
            d = load_timeseries(run, fname, wide=True)
            if d is None:
                mark_missing(ax, fname)
                continue
            d = d.loc[:, (d != 0).any()]  # only taxa with uptake
            active |= set(d.columns)
            for tax in d.columns:
                ax.plot(d.index, d[tax], color=color(tax), lw=1)
        row[0].set_ylabel(f"{fname.split('_')[0]} uptake\n(C per day)")
        handles = [Line2D([], [], color=color(t), label=t) for t in sorted(active, key=natural_key)]
        if handles:
            row[-1].legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
                           fontsize="x-small", ncol=max(1, len(handles) // 15 + 1))
    fig.suptitle("VOC uptake per taxon")
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    save(fig, "VOC_uptake_TaxonSeries")
    plt.show()


plot_taxon_uptake()

In [ ]:
if SAVE_PLOTS and PLOT_DIR.exists():
    (PLOT_DIR / "runs.txt").write_text("\n".join(f"{r}\t{label(r)}" for r in RUNS) + "\n")
    print("Figures saved to", PLOT_DIR.resolve())